In [ ]:
import os, urllib.request, zipfile, io, json, time
import pandas as pd
from pathlib import Path

RAW = "data/raw"
os.makedirs(RAW, exist_ok=True)
print(f"Raw data directory: {os.path.abspath(RAW)}")


## 1. APL — Accessibilité Potentielle Localisée aux MG (DREES)

Source: DREES data portal. XLSX with 3 sheets (Paramètres, APL 2022, APL 2023). ~35K communes.

In [ ]:
APL_PATH = f"{RAW}/apl.xlsx"
APL_URL = "https://data.drees.solidarites-sante.gouv.fr/api/v2/catalog/datasets/530_l-accessibilite-potentielle-localisee-apl/attachments/indicateur_d_accessibilite_potentielle_localisee_apl_aux_medecins_generalistes_xlsx"
APL_MIN_SIZE = 1_000_000  # 1 MB

try:
    if os.path.exists(APL_PATH) and os.path.getsize(APL_PATH) > APL_MIN_SIZE:
        print(f"APL: Already downloaded ({os.path.getsize(APL_PATH):,} bytes)")
    else:
        print("Downloading APL (DREES XLSX)...")
        req = urllib.request.Request(APL_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = resp.read()
        with open(APL_PATH, "wb") as f:
            f.write(data)
        print(f"APL: Downloaded ({os.path.getsize(APL_PATH):,} bytes)")

    apl_xl = pd.ExcelFile(APL_PATH)
    print(f"APL sheets: {apl_xl.sheet_names}")
    apl_all = apl_xl.parse(apl_xl.sheet_names[-1])
    print(f"APL rows (last sheet): {len(apl_all):,}")
    print(f"APL columns: {list(apl_all.columns[:5])}")
    assert len(apl_all) > 30_000, f"APL too few rows: {len(apl_all)}"
    cols_lower = [str(c).lower() for c in apl_all.columns]
    assert any("commune" in c or "apl" in c or "com" == c or "codgeo" in c for c in cols_lower),         f"APL missing commune/apl col: {list(apl_all.columns)}"
    print("APL: OK")
except Exception as e:
    print(f"APL ERROR: {e}"); import traceback; traceback.print_exc()


## 2. RPPS — Annuaire Santé (Professionnels de Santé)

Source: data.gouv.fr. Pipe-separated TXT, ~801 MB. 2.2M rows. Use urlretrieve.

In [ ]:
RPPS_PATH = f"{RAW}/rpps.csv"
RPPS_URL = "https://static.data.gouv.fr/resources/annuaire-sante-extractions-des-donnees-en-libre-acces-des-professionnels-intervenant-dans-le-systeme-de-sante-rpps/20260407-074042/ps-libreacces-personne-activite.txt"
RPPS_MIN_SIZE = 100_000_000  # 100 MB

try:
    if os.path.exists(RPPS_PATH) and os.path.getsize(RPPS_PATH) > RPPS_MIN_SIZE:
        print(f"RPPS: Already downloaded ({os.path.getsize(RPPS_PATH):,} bytes)")
    else:
        print("Downloading RPPS (~801 MB, several minutes)...")
        def reporthook(count, block_size, total_size):
            if total_size > 0 and count % 2000 == 0:
                pct = min(count * block_size / total_size * 100, 100)
                print(f"  {pct:.1f}%", end="\r")
        urllib.request.urlretrieve(RPPS_URL, RPPS_PATH, reporthook=reporthook)
        print(f"\nRPPS: Downloaded ({os.path.getsize(RPPS_PATH):,} bytes)")

    rpps_size = os.path.getsize(RPPS_PATH)
    assert rpps_size > RPPS_MIN_SIZE, f"RPPS too small: {rpps_size:,} bytes"

    for enc in ("utf-8", "latin-1"):
        try:
            rpps_sample = pd.read_csv(RPPS_PATH, nrows=5, encoding=enc, sep="|", low_memory=False)
            rpps_encoding = enc; break
        except UnicodeDecodeError:
            continue
    print(f"RPPS encoding: {rpps_encoding}")
    print(f"RPPS columns: {list(rpps_sample.columns[:5])}")

    with open(RPPS_PATH, encoding=rpps_encoding, errors="replace") as f:
        rpps_rows = sum(1 for _ in f) - 1
    print(f"RPPS rows: {rpps_rows:,}")
    assert rpps_rows > 1_000_000, f"RPPS too few rows: {rpps_rows:,}"
    print("RPPS: OK")
except Exception as e:
    print(f"RPPS ERROR: {e}"); import traceback; traceback.print_exc()


## 3. FINESS — Fichier des Établissements de Santé

Source: static.data.gouv.fr. Semicolon-separated CSV ~36MB, 102K rows. Header row = metadata line (skip it). Column 18 = category code, column 19 = category label.

In [ ]:
FINESS_PATH = f"{RAW}/finess.csv"
FINESS_URL = "https://static.data.gouv.fr/resources/finess-extraction-du-fichier-des-etablissements/20260312-094813/etalab-cs1100502-stock-20260311-0344.csv"
FINESS_MIN_SIZE = 5_000_000

try:
    if os.path.exists(FINESS_PATH) and os.path.getsize(FINESS_PATH) > FINESS_MIN_SIZE:
        print(f"FINESS: Already downloaded ({os.path.getsize(FINESS_PATH):,} bytes)")
    else:
        print("Downloading FINESS...")
        urllib.request.urlretrieve(FINESS_URL, FINESS_PATH)
        print(f"FINESS: Downloaded ({os.path.getsize(FINESS_PATH):,} bytes)")

    # File has 1 metadata header line then data rows (no column headers)
    # skiprows=1 to skip "finess;etalab;N;DATE" metadata line
    finess_raw = pd.read_csv(FINESS_PATH, sep=";", encoding="utf-8",
                              skiprows=1, header=None, low_memory=False, dtype=str)
    print(f"FINESS rows: {len(finess_raw):,}, columns: {len(finess_raw.columns)}")
    # Col 0=type, 1=nofinessET, 3=rs, 13=dept, 14=nomDept, 15=cp_commune
    # Col 18=catCode, 19=catLabel
    sample_row = finess_raw.iloc[0]
    print(f"  Sample: type={sample_row[0]}, finess={sample_row[1]}, rs={sample_row[3]}, dept={sample_row[13]}")
    assert len(finess_raw) > 50_000, f"FINESS too few rows: {len(finess_raw)}"
    print("FINESS: OK")
except Exception as e:
    print(f"FINESS ERROR: {e}"); import traceback; traceback.print_exc()


## 4. MSP — Maisons de Santé Pluriprofessionnelles

Extracted from FINESS using category code 603 ('Maison de santé L.6223-3'). ~3,080 MSP nationally.

In [ ]:
MSP_PATH = f"{RAW}/msp.csv"

try:
    if os.path.exists(MSP_PATH) and os.path.getsize(MSP_PATH) > 10_000:
        print(f"MSP: Already available ({os.path.getsize(MSP_PATH):,} bytes)")
    else:
        assert os.path.exists(FINESS_PATH), "FINESS must be downloaded first (run Cell 3)"
        print("MSP: Extracting from FINESS (category 603 = Maison de santé L.6223-3)...")
        finess_raw = pd.read_csv(FINESS_PATH, sep=";", encoding="utf-8",
                                  skiprows=1, header=None, low_memory=False, dtype=str)

        # Column 18 = category code, 19 = category label
        msp_mask = finess_raw[18].astype(str).str.strip() == "603"
        msp_df = finess_raw[msp_mask][[0, 1, 3, 4, 13, 14, 15, 18, 19]].copy()
        msp_df.columns = ["type", "nofinessET", "rs", "rslongue",
                           "codeDept", "nomDept", "cp_commune", "catCode", "catLabel"]
        msp_df.to_csv(MSP_PATH, index=False, encoding="utf-8")
        print(f"  Saved {len(msp_df)} MSP rows (cat 603) to {MSP_PATH}")

    msp_df = pd.read_csv(MSP_PATH)
    print(f"MSP columns: {list(msp_df.columns)}")
    msp_rows = len(msp_df)
    print(f"MSP rows: {msp_rows:,}")
    assert msp_rows > 1_000, f"MSP too few rows: {msp_rows:,} (expected >1K)"
    print("MSP: OK")
except Exception as e:
    print(f"MSP ERROR: {e}"); import traceback; traceback.print_exc()


## 5. INSEE Population par Commune

Source: Geo API gouv.fr — 34,969 communes with code, nom, codeDepartement, codeRegion, codesPostaux, population. INSEE.fr direct downloads have intermittent 500 errors.

In [ ]:
INSEE_POP_PATH = f"{RAW}/insee_pop.csv"
GEO_API_URL = "https://geo.api.gouv.fr/communes?fields=code,nom,codeDepartement,codeRegion,codesPostaux,population&format=json"
INSEE_POP_MIN_SIZE = 500_000

try:
    if os.path.exists(INSEE_POP_PATH) and os.path.getsize(INSEE_POP_PATH) > INSEE_POP_MIN_SIZE:
        print(f"INSEE Pop: Already downloaded ({os.path.getsize(INSEE_POP_PATH):,} bytes)")
    else:
        print("Downloading INSEE Population (Geo API)...")
        req = urllib.request.Request(GEO_API_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            communes_json = json.loads(resp.read().decode("utf-8"))
        print(f"  Received {len(communes_json)} communes")
        for c in communes_json:
            if isinstance(c.get("codesPostaux"), list):
                c["codesPostaux"] = ",".join(c["codesPostaux"])
        pop_df = pd.DataFrame(communes_json)
        pop_df.to_csv(INSEE_POP_PATH, index=False, encoding="utf-8")

    pop_df = pd.read_csv(INSEE_POP_PATH)
    print(f"INSEE Pop columns: {list(pop_df.columns)}")
    pop_rows = len(pop_df)
    print(f"INSEE Pop rows: {pop_rows:,}")
    assert pop_rows > 34_000, f"INSEE Pop too few rows: {pop_rows:,}"
    assert "code" in pop_df.columns, "Missing 'code' column"
    print("INSEE Pop: OK")
except Exception as e:
    print(f"INSEE Pop ERROR: {e}"); import traceback; traceback.print_exc()


## 6. FiLoSoFi 2018 — Revenus et Pauvreté par Commune (INSEE)

Source: INSEE statistiques. ZIP with CSV at commune level. ~34,928 communes.
INSEE infrastructure restored April 2026. URL: https://www.insee.fr/fr/statistiques/5009236

In [ ]:
FILOSOFI_ZIP = f"{RAW}/filosofi.zip"
FILOSOFI_CSV = f"{RAW}/cc_filosofi_2018_COM-geo2021.CSV"
FILOSOFI_URL = "https://www.insee.fr/fr/statistiques/fichier/5009236/base-cc-filosofi-2018_CSV_geo2021.zip"
FILOSOFI_ZIP_MIN_SIZE = 500_000

try:
    # FiLoSoFi now available — INSEE infrastructure restored April 2026
    if os.path.exists(FILOSOFI_ZIP) and os.path.getsize(FILOSOFI_ZIP) > FILOSOFI_ZIP_MIN_SIZE:
        print(f"FiLoSoFi: Already downloaded ({os.path.getsize(FILOSOFI_ZIP):,} bytes)")
    else:
        print("Downloading FiLoSoFi 2018 (INSEE commune-level poverty data)...")
        def reporthook_filo(count, block_size, total_size):
            if total_size > 0 and count % 200 == 0:
                pct = min(count * block_size / total_size * 100, 100)
                print(f"  {pct:.1f}%", end="\r")
        urllib.request.urlretrieve(FILOSOFI_URL, FILOSOFI_ZIP, reporthook=reporthook_filo)
        print(f"\nFiLoSoFi: Downloaded ({os.path.getsize(FILOSOFI_ZIP):,} bytes)")

    assert os.path.getsize(FILOSOFI_ZIP) > FILOSOFI_ZIP_MIN_SIZE, f"FiLoSoFi zip too small: {os.path.getsize(FILOSOFI_ZIP):,} bytes"

    # Extract commune-level CSV if needed
    if not os.path.exists(FILOSOFI_CSV):
        with zipfile.ZipFile(FILOSOFI_ZIP) as zf:
            zf.extract("cc_filosofi_2018_COM-geo2021.CSV", RAW)
        print(f"FiLoSoFi: Extracted cc_filosofi_2018_COM-geo2021.CSV")

    filo_df = pd.read_csv(FILOSOFI_CSV, sep=";", dtype={"CODGEO": str})
    assert len(filo_df) > 34000, f"FiLoSoFi too few rows: {len(filo_df)}"
    assert "TP6018" in filo_df.columns, f"TP6018 column missing. Cols: {filo_df.columns.tolist()[:10]}"
    assert "MED18" in filo_df.columns, f"MED18 column missing. Cols: {filo_df.columns.tolist()[:10]}"

    print(f"FiLoSoFi rows: {len(filo_df):,}")
    print(f"TP6018 (poverty rate) non-null: {filo_df['TP6018'].notna().sum():,} ({filo_df['TP6018'].notna().mean():.1%}) — statistical secrecy for small communes")
    print(f"MED18 (median income) non-null: {filo_df['MED18'].notna().sum():,} ({filo_df['MED18'].notna().mean():.1%})")

    # Delete placeholder if present
    placeholder = f"{RAW}/filosofi_placeholder.csv"
    if os.path.exists(placeholder):
        os.remove(placeholder)
        print(f"Deleted: {placeholder}")

    print("FiLoSoFi: OK")
except Exception as e:
    print(f"FiLoSoFi ERROR: {e}"); import traceback; traceback.print_exc()


## 6b. RP2020 — Population par structure d'âge (INSEE)

Source: INSEE recensement 2020. ZIP with CSV ~117MB unzipped. 34,973 communes.
Columns used: CODGEO, P20_POP (total), P20_POP7589 (75-89), P20_POP90P (90+).

In [ ]:
POP_AGE_ZIP = f"{RAW}/pop_age.zip"
POP_AGE_CSV = f"{RAW}/base-cc-evol-struct-pop-2020.CSV"
POP_AGE_URL = "https://www.insee.fr/fr/statistiques/fichier/7632446/base-cc-evol-struct-pop-2020_csv.zip"
POP_AGE_MIN_SIZE = 10_000_000  # ~39MB compressed

try:
    if os.path.exists(POP_AGE_ZIP) and os.path.getsize(POP_AGE_ZIP) > POP_AGE_MIN_SIZE:
        print(f"RP2020: Already downloaded ({os.path.getsize(POP_AGE_ZIP):,} bytes)")
    else:
        print("Downloading RP2020 age structure (~39MB compressed, 117MB unzipped)...")
        def reporthook_pop(count, block_size, total_size):
            if total_size > 0 and count % 500 == 0:
                pct = min(count * block_size / total_size * 100, 100)
                print(f"  {pct:.1f}%", end="\r")
        urllib.request.urlretrieve(POP_AGE_URL, POP_AGE_ZIP, reporthook=reporthook_pop)
        print(f"\nRP2020: Downloaded ({os.path.getsize(POP_AGE_ZIP):,} bytes)")

    assert os.path.getsize(POP_AGE_ZIP) > POP_AGE_MIN_SIZE, f"RP2020 zip too small: {os.path.getsize(POP_AGE_ZIP):,} bytes"

    # Extract if needed (large file — 117MB unzipped)
    if not os.path.exists(POP_AGE_CSV):
        with zipfile.ZipFile(POP_AGE_ZIP) as zf:
            zf.extract("base-cc-evol-struct-pop-2020.CSV", RAW)
        print(f"RP2020: Extracted base-cc-evol-struct-pop-2020.CSV")

    # Read only needed columns (full file has 100+ cols)
    pop_age_df = pd.read_csv(
        POP_AGE_CSV, sep=";", dtype={"CODGEO": str},
        usecols=["CODGEO", "P20_POP", "P20_POP7589", "P20_POP90P"]
    )

    assert len(pop_age_df) > 34000, f"RP2020 too few rows: {len(pop_age_df)}"
    assert "P20_POP7589" in pop_age_df.columns, "P20_POP7589 missing"
    assert "P20_POP90P" in pop_age_df.columns, "P20_POP90P missing"

    print(f"RP2020 rows: {len(pop_age_df):,}")
    print(f"P20_POP7589 non-null: {pop_age_df['P20_POP7589'].notna().sum():,} ({pop_age_df['P20_POP7589'].notna().mean():.1%})")
    print(f"P20_POP90P non-null: {pop_age_df['P20_POP90P'].notna().sum():,} ({pop_age_df['P20_POP90P'].notna().mean():.1%})")
    print("RP2020: OK")
except Exception as e:
    print(f"RP2020 ERROR: {e}"); import traceback; traceback.print_exc()


## 7. Pathologies — Data Ameli (5.2M rows, par département/âge/pathologie)

In [ ]:
PATHO_PATH = f"{RAW}/pathologies.csv"
PATHO_URL = "https://data.ameli.fr/api/explore/v2.1/catalog/datasets/effectifs/exports/csv?lang=fr&timezone=Europe%2FParis&use_labels=true&delimiter=%3B"
PATHO_MIN_SIZE = 100_000

try:
    if os.path.exists(PATHO_PATH) and os.path.getsize(PATHO_PATH) > PATHO_MIN_SIZE:
        print(f"Pathologies: Already downloaded ({os.path.getsize(PATHO_PATH):,} bytes)")
    else:
        print("Downloading Pathologies (data.ameli — may take several minutes)...")
        req = urllib.request.Request(PATHO_URL, headers={"User-Agent": "Mozilla/5.0", "Accept": "text/csv,*/*"})
        with urllib.request.urlopen(req, timeout=300) as resp:
            data = resp.read()
        with open(PATHO_PATH, "wb") as f:
            f.write(data)
        print(f"Pathologies: Downloaded ({os.path.getsize(PATHO_PATH):,} bytes)")

    patho_df = pd.read_csv(PATHO_PATH, sep=";", encoding="utf-8", nrows=5)
    print(f"Pathologies columns: {list(patho_df.columns[:5])}")
    with open(PATHO_PATH, encoding="utf-8", errors="replace") as f:
        patho_rows = sum(1 for _ in f) - 1
    print(f"Pathologies rows: {patho_rows:,}")
    assert patho_rows > 50, f"Pathologies too few rows: {patho_rows:,}"
    print("Pathologies: OK")
except Exception as e:
    print(f"Pathologies ERROR: {e}"); import traceback; traceback.print_exc()


## 8. Admin Express — GeoJSON Communes 2025 (IGN/data.gouv.fr)

44MB file. a-com2025.geojson from Admin Express COG PLUS 2025.

In [ ]:
GEO_PATH = f"{RAW}/communes_geo.geojson"
GEO_URL = "https://static.data.gouv.fr/resources/communes-cantons-et-epci-2025-admin-express-cog-plus-ign/20250626-075329/a-com2025.geojson"
GEO_MIN_SIZE = 1_000_000

try:
    if os.path.exists(GEO_PATH) and os.path.getsize(GEO_PATH) > GEO_MIN_SIZE:
        print(f"Admin Express: Already downloaded ({os.path.getsize(GEO_PATH):,} bytes)")
    else:
        print("Downloading Admin Express GeoJSON (~44 MB)...")
        def reporthook(count, block_size, total_size):
            if total_size > 0 and count % 500 == 0:
                pct = min(count * block_size / total_size * 100, 100)
                print(f"  {pct:.1f}%", end="\r")
        urllib.request.urlretrieve(GEO_URL, GEO_PATH, reporthook=reporthook)
        print(f"\nAdmin Express: Downloaded ({os.path.getsize(GEO_PATH):,} bytes)")

    geo_size = os.path.getsize(GEO_PATH)
    assert geo_size > GEO_MIN_SIZE, f"Too small: {geo_size:,} bytes"
    with open(GEO_PATH, "r", encoding="utf-8", errors="replace") as f:
        first = f.read(100)
    assert "{" in first, f"Not valid GeoJSON: {first[:50]}"
    print(f"Admin Express: OK ({geo_size:,} bytes)")
except Exception as e:
    print(f"Admin Express ERROR: {e}"); import traceback; traceback.print_exc()


## 9. Diagnostic Accès Soins Urgents (DREES 2019)

XLSX with 2 sheets (LISEZ-MOI, BASECOM_URGENCES_2019). Data starts at row 6 (skiprows=5). 35,010 communes.

In [ ]:
URGENCES_PATH = f"{RAW}/urgences.xlsx"
URGENCES_URL = "https://data.drees.solidarites-sante.gouv.fr/api/v2/catalog/datasets/2943_diagnostic-d-acces-aux-soins-urgents/attachments/diagnostic_d_acces_aux_soins_urgents_au_31_12_2019_xlsx"
URGENCES_MIN_SIZE = 500_000

try:
    if os.path.exists(URGENCES_PATH) and os.path.getsize(URGENCES_PATH) > URGENCES_MIN_SIZE:
        print(f"Urgences: Already downloaded ({os.path.getsize(URGENCES_PATH):,} bytes)")
    else:
        print("Downloading Urgences XLSX (DREES 2019)...")
        req = urllib.request.Request(URGENCES_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = resp.read()
        with open(URGENCES_PATH, "wb") as f:
            f.write(data)
        print(f"Urgences: Downloaded ({os.path.getsize(URGENCES_PATH):,} bytes)")

    urg_xl = pd.ExcelFile(URGENCES_PATH)
    print(f"Urgences sheets: {urg_xl.sheet_names}")
    # Data sheet: BASECOM_URGENCES_2019, actual data starts at row 6 (skiprows=5)
    DATA_SHEET = "BASECOM_URGENCES_2019"
    urg_all = urg_xl.parse(DATA_SHEET, skiprows=5)
    print(f"Urgences columns: {list(urg_all.columns[:8])}")
    urg_rows = len(urg_all)
    print(f"Urgences rows: {urg_rows:,}")
    assert urg_rows > 30_000, f"Urgences too few rows: {urg_rows:,}"
    cols_lower = [str(c).lower() for c in urg_all.columns]
    time_cols = [c for c in urg_all.columns if any(t in str(c).lower()
                 for t in ["temps", "min", "acces", "tps", "duree", "dist"])]
    assert len(time_cols) > 0, f"Urgences missing time col: {list(urg_all.columns)}"
    print(f"Time columns: {time_cols}")
    print("Urgences: OK")
except Exception as e:
    print(f"Urgences ERROR: {e}"); import traceback; traceback.print_exc()


## 10. La Poste — Table Codes Postaux → Communes INSEE

Source: datanova.laposte.fr. 39,191 rows. Comment line with column headers, skip it and assign manually. Encoding: latin-1.

In [ ]:
LAPOSTE_PATH = f"{RAW}/la_poste_cp_commune.csv"
LAPOSTE_URL = "https://datanova.laposte.fr/data-fair/api/v1/datasets/laposte-hexasmal/raw"
LAPOSTE_MIN_SIZE = 500_000

try:
    if os.path.exists(LAPOSTE_PATH) and os.path.getsize(LAPOSTE_PATH) > LAPOSTE_MIN_SIZE:
        print(f"La Poste: Already downloaded ({os.path.getsize(LAPOSTE_PATH):,} bytes)")
    else:
        print("Downloading La Poste CP→Commune table...")
        req = urllib.request.Request(LAPOSTE_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = resp.read()
        # File has comment '#Code_commune_INSEE;...' header, then data
        # Re-parse with latin-1, skip comment, assign proper column names
        import io as _io
        raw_lines = data.decode("latin-1").split("\n")
        # Filter out comment lines (start with #)
        data_lines = [l for l in raw_lines if not l.startswith("#")]
        content = "\n".join(data_lines)
        lp_df = pd.read_csv(_io.StringIO(content), sep=";", header=None)
        lp_df.columns = ["Code_commune_INSEE", "Nom_commune", "Code_postal",
                          "Libelle_acheminement", "Ligne_5"][:len(lp_df.columns)]
        lp_df.to_csv(LAPOSTE_PATH, index=False, encoding="utf-8")
        print(f"La Poste: Saved {len(lp_df)} rows ({os.path.getsize(LAPOSTE_PATH):,} bytes)")

    lp_df = pd.read_csv(LAPOSTE_PATH, encoding="utf-8")
    print(f"La Poste columns: {list(lp_df.columns)}")
    lp_rows = len(lp_df)
    print(f"La Poste rows: {lp_rows:,}")
    assert lp_rows > 30_000, f"La Poste too few rows: {lp_rows:,}"
    cols_lower = [c.lower() for c in lp_df.columns]
    assert any("code_postal" in c or "postal" in c for c in cols_lower),         f"La Poste missing postal code col: {list(lp_df.columns)}"
    assert any("commune" in c or "insee" in c for c in cols_lower),         f"La Poste missing commune col: {list(lp_df.columns)}"
    print("La Poste: OK")
except Exception as e:
    print(f"La Poste ERROR: {e}"); import traceback; traceback.print_exc()


## Summary — Statut de tous les fichiers

In [ ]:
import os

datasets = [
    ("APL",             f"{RAW}/apl.xlsx",                   1_000_000),
    ("RPPS",            f"{RAW}/rpps.csv",                 100_000_000),
    ("FINESS",          f"{RAW}/finess.csv",                 5_000_000),
    ("MSP",             f"{RAW}/msp.csv",                       10_000),
    ("INSEE Pop",       f"{RAW}/insee_pop.csv",               500_000),
    ("FiLoSoFi zip",    f"{RAW}/filosofi.zip",                500_000),
    ("RP2020 Age zip",  f"{RAW}/pop_age.zip",             10_000_000),
    ("Pathologies",     f"{RAW}/pathologies.csv",             100_000),
    ("Admin Express",   f"{RAW}/communes_geo.geojson",      1_000_000),
    ("Urgences",        f"{RAW}/urgences.xlsx",               500_000),
    ("La Poste",        f"{RAW}/la_poste_cp_commune.csv",    500_000),
]

print(f"{'Dataset':<22} {'File':<35} {'Size (MB)':>10} {'Status'}")
print("-" * 85)
found_count = 0
for name, path, min_size in datasets:
    if os.path.exists(path):
        size = os.path.getsize(path)
        size_mb = size / 1_000_000
        status = "OK" if size >= min_size else f"SMALL ({size:,}B)"
        found_count += 1
    else:
        size_mb = 0; status = "MISSING"
    print(f"{name:<22} {os.path.basename(path):<35} {size_mb:>10.2f}  {status}")

# Critical assertions
critical = [
    f"{RAW}/apl.xlsx",
    f"{RAW}/rpps.csv",
    f"{RAW}/finess.csv",
    f"{RAW}/msp.csv",
    f"{RAW}/insee_pop.csv",
    f"{RAW}/filosofi.zip",
    f"{RAW}/pop_age.zip",
    f"{RAW}/pathologies.csv",
    f"{RAW}/communes_geo.geojson",
    f"{RAW}/urgences.xlsx",
    f"{RAW}/la_poste_cp_commune.csv",
]
missing = [p for p in critical if not os.path.exists(p)]
assert len(missing) == 0, f"Critical files missing: {missing}"

print()
print(f"Critical files: {len(critical) - len(missing)}/{len(critical)} present.")
print(f"FiLoSoFi: zip present (INSEE infrastructure restored April 2026)")
print(f"RP2020 age structure: zip present")
print()
print("DOWNLOAD PIPELINE COMPLETE")
